# RetentionAI — 02. Data Understanding

**Objective:** understand what this dataset actually *is* — grain, target, dtypes, real quality quirks, and structural relationships between columns — before any EDA plots or modeling decisions.

No hypotheses about churn patterns yet. No feature engineering yet. Just: what does this data literally contain, and can we trust it.

> Run on `data/sample_synthetic.csv` in this environment since the real Kaggle CSV isn't available here. Re-run this notebook yourself on `data/raw/telco_churn.csv` — the code doesn't change, only the numbers will.....

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"
SYNTHETIC_PATH = PROJECT_ROOT / "data" / "sample_synthetic.csv"

DATA_PATH = RAW_PATH if RAW_PATH.exists() else SYNTHETIC_PATH
print(f"Loading: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df.head(3)

Loading: /home/claude/retentionai/data/sample_synthetic.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,SYN-00000,Female,0,Yes,Yes,1,Yes,Yes,DSL,No,...,No,No,Yes,No,Month-to-month,No,Mailed check,65.51,87.37,Yes
1,SYN-00001,Female,0,Yes,No,46,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),80.49,3720.65,Yes
2,SYN-00002,Female,0,Yes,No,32,Yes,No,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,75.07,2365.03,Yes


## 1. Grain — what does one row actually represent?

Claim: one row = one customer, uniquely identified by `customerID`. Verify it, don't assume it.

In [2]:
print("Shape:", df.shape)
print("Unique customerID:", df['customerID'].nunique())
print("Duplicated customerID rows:", df['customerID'].duplicated().sum())

assert df['customerID'].nunique() == len(df), "customerID is not a unique row identifier — grain assumption is wrong!"
print("\nGrain confirmed: one row = one customer.")

Shape: (1500, 21)
Unique customerID: 1500
Duplicated customerID rows: 0

Grain confirmed: one row = one customer.


## 2. Target — Churn, exactly as it is (not the ~26% we assumed back in Stage 1)

Stage 1's cost-sensitive threshold math used an assumed ~26% churn rate. That assumption gets checked here, against real numbers, not re-derived from it — this is verification, not re-framing.

In [3]:
print(df['Churn'].value_counts())
print()
print(df['Churn'].value_counts(normalize=True).round(4))

Churn
No     1283
Yes     217
Name: count, dtype: int64

Churn
No     0.8553
Yes    0.1447
Name: proportion, dtype: float64


## 3. Raw dtypes — spot the quirk before it breaks something later

`TotalCharges` should be a number. Check what pandas actually inferred.

In [4]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

`TotalCharges` came in as text, not a number. Before assuming this means missing data, check for actual `NaN`s first — a blank string (`" "`) and a real `NaN` are not the same thing to pandas, and confusing them leads to code that silently does nothing.

In [5]:
print("Real NaNs across the whole dataframe:", df.isnull().sum().sum())

Real NaNs across the whole dataframe: 0


Zero real `NaN`s — the missing values are hiding as blank strings, not flagged by pandas as missing at all. That's exactly why a naive `df.isnull().sum()` check alone would have missed this data quality issue entirely.

## 4. Fixing TotalCharges — a dtype correction, not a modeling decision

`errors='coerce'` turns anything non-numeric (including blank strings) into a real `NaN`, which we can then actually count and reason about.

**Why fill with 0, and not the column mean (a common default)?** Because filling with the mean would be *lying* about these specific customers — the mean assumes "we don't know this value, so our best guess is the average." But we *do* know why it's blank: every one of these rows is a customer with `tenure == 0`, meaning they've been billed exactly $0 in total charges so far, because they haven't reached their first billing cycle yet. That's not an unknown value to estimate — it's a known value we can state exactly. Using the mean here would inject a fake ~$2000 charge onto brand-new customers who have genuinely paid nothing yet, which would quietly bias any model trained on this column.

In [6]:
coerced = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Became NaN after coercion:", coerced.isnull().sum())
print("Tenure values for those rows:", sorted(df.loc[coerced.isnull(), 'tenure'].unique()))

df['TotalCharges'] = coerced.fillna(0)
print("\nFixed. Remaining NaNs in TotalCharges:", df['TotalCharges'].isnull().sum())

Became NaN after coercion: 17
Tenure values for those rows: [np.int64(0)]

Fixed. Remaining NaNs in TotalCharges: 0


## 5. Structural dependencies — is `"No internet service"` really *missing* data?

Several columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) show **three** values, not two: `Yes`, `No`, and `No internet service`. That third value only makes logical sense for customers whose `InternetService` is `"No"` — you can't have online security for internet you don't have. If that's true, it isn't missing data at all — it's a real, structurally determined fact about the customer. Verify it directly instead of assuming it.

In [7]:
pd.crosstab(df['InternetService'], df['OnlineSecurity'])

OnlineSecurity,No,No internet service,Yes
InternetService,,,
DSL,323,0,189
Fiber optic,438,0,224
No,0,326,0


Confirmed — `"No internet service"` appears in exactly one row of this table: where `InternetService == "No"`, and nowhere else. It's zero missing data; it's a perfectly determined structural fact. The same logic applies to `MultipleLines`, which shows `"No phone service"` only when `PhoneService == "No"`:

In [8]:
pd.crosstab(df['PhoneService'], df['MultipleLines'])

MultipleLines,No,No phone service,Yes
PhoneService,,,
No,0,168,0
Yes,718,0,614


**Why this matters beyond curiosity:** these add-on columns are not independent features — six of them (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) all collapse to the same "No internet service" state together, driven entirely by one underlying column (`InternetService`). That's a strong hint of redundancy worth remembering once we get to feature engineering and the VIF check — six columns that all move together for the same ~22% of customers can inflate multicollinearity in a way that isn't obvious just from looking at any single column alone.

## 6. The SeniorCitizen inconsistency

`SeniorCitizen` is conceptually the same kind of fact as `Partner` or `Dependents` — a yes/no fact about the customer — but check how each is actually stored:

In [9]:
print("SeniorCitizen unique values:", sorted(df['SeniorCitizen'].unique()))
print("Partner unique values:       ", sorted(df['Partner'].unique()))

SeniorCitizen unique values: [np.int64(0), np.int64(1)]
Partner unique values:        ['No', 'Yes']


`SeniorCitizen` is stored as `0`/`1` (integer), while `Partner` and `Dependents` are stored as `"Yes"`/`"No"` (text) — inconsistent encoding for the same *kind* of information. This almost certainly traces back to how the source survey/database captured the field (a checkbox stored as a bit flag vs. a dropdown stored as its label) rather than any meaningful difference between the concepts.

**This is a Stage 3 observation, not a Stage 3 fix.** Making all binary columns consistent (e.g., mapping every yes/no field to 0/1) is an encoding decision that belongs to the modeling pipeline (Stage 6), where it can be done for every categorical column at once, deliberately, and logged as its own decision — not patched ad hoc the moment it's spotted.

## 7. Data Dictionary

Grounded in the dataset's own documented meaning (IBM's original Telco Customer Churn sample data, as published on Kaggle), cross-checked against what's actually observed above.

| Column | Meaning | Type | Role |
|---|---|---|---|
| customerID | Unique alphanumeric ID per customer | Identifier | ID (drop before modeling) |
| gender | Male / Female | Categorical | Feature |
| SeniorCitizen | Whether the customer is a senior citizen | Binary (0/1 — inconsistent encoding, see §6) | Feature |
| Partner | Whether the customer has a partner | Binary (Yes/No) | Feature |
| Dependents | Whether the customer has dependents | Binary (Yes/No) | Feature |
| tenure | Months the customer has stayed with the company | Numeric | Feature |
| PhoneService | Whether the customer has phone service | Binary (Yes/No) | Feature |
| MultipleLines | Whether the customer has multiple phone lines | Categorical (Yes/No/No phone service) | Feature — structurally tied to PhoneService |
| InternetService | Type of internet service | Categorical (DSL/Fiber optic/No) | Feature |
| OnlineSecurity | Online security add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| OnlineBackup | Online backup add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| DeviceProtection | Device protection add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| TechSupport | Tech support add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| StreamingTV | Streaming TV add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| StreamingMovies | Streaming movies add-on subscribed | Categorical (Yes/No/No internet service) | Feature — structurally tied to InternetService |
| Contract | Contract term | Categorical (Month-to-month/One year/Two year) | Feature |
| PaperlessBilling | Whether billing is paperless | Binary (Yes/No) | Feature |
| PaymentMethod | How the customer pays | Categorical (4 methods) | Feature |
| MonthlyCharges | Current monthly charge amount | Numeric | Feature |
| TotalCharges | Total amount charged over the customer's tenure | Numeric (stored as text with blanks — fixed in §4) | Feature |
| Churn | Whether the customer left within the last billing period | Binary (Yes/No) | **Target** |

**On the real Kaggle data (not the synthetic placeholder above), expect the churn rate to land close to the widely-reported ~26.5% for this exact dataset (5174 stayed / 1869 churned out of 7043) — run §2 yourself on the real file and confirm this matches, since this notebook only has the synthetic version to run on here.**

## Stage 3 summary — quirks found, decisions made

- **Grain confirmed:** one row = one customer, `customerID` is a true unique key.
- **TotalCharges** arrives as text due to blank-string values for zero-tenure customers — fixed via `coerce` + `fillna(0)`, because these are known $0 values, not unknown ones (filling with the mean would have been factually wrong here, not just a stylistic choice).
- **`"No internet service"` / `"No phone service"`** are not missing data — they're structurally determined by `InternetService` / `PhoneService`, confirmed empirically via crosstabs, not assumed.
- **Six add-on columns move together** with `InternetService` — a multicollinearity risk flagged now, to check for real with VIF once we reach the modeling pipeline.
- **`SeniorCitizen`'s 0/1 encoding**, inconsistent with other Yes/No columns, is noted here but deliberately left unfixed — that's a Stage 6 pipeline decision, not a Stage 3 one.

Stage 3 is done. Next: Stage 4 — EDA, now that we actually know what we're looking at.